# DSAR × Lakeflow Declarative Pipelines · 02 · Erasure (all subjects, all layers)

Processes the **`dsar_request`** queue. For **every PENDING subject** it erases the
subject's records from **every layer** — `raw_user`, `bronze_user`, `silver_user`
(base tables) — then **refreshes** the `gold_user` materialized view, physically
purges, validates no trace, and marks the request COMPLETE.

This is the same erasure behavior as the base [`dsar_erasure/`](../../dsar_erasure)
layer, adapted so it runs **against a live streaming pipeline with zero downtime**:
because `01`/`01b` set `skipChangeCommits` on every streaming read, each DELETE is
skipped by the streams instead of crashing them.

### How subjects are matched

Intake gives an **email**. Bronze/silver have already redacted the email, so we
resolve **email → stable `user_id`** from `raw_user` once per subject, then erase by
`user_id` at every layer. (`user_id` is a non-PII key that survives masking.)

### Two request types (per row in `dsar_request`)

- **DELETE** — remove the subject's rows entirely.
- **OBFUSCATE** — keep rows, redact PII cells (only `raw_user` still holds cleartext
  PII; downstream is already masked).

### Gold is a materialized view

You cannot `DELETE` from a view. We erase the **base tables** and **refresh** the
gold MV so it re-derives clean from the erased silver.

## 0. Configuration

In [ ]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", "dkushari_uc", "1 Catalog")
dbutils.widgets.text("schema", "allegiant_air_sdp_dsar", "2 Schema (match the pipeline target)")
dbutils.widgets.text("redaction_token", "***REDACTED***", "3 Redaction token")
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"], "4 Dry run (preview only)")
dbutils.widgets.dropdown("do_purge", "true", ["true", "false"], "5 Physically VACUUM after erase")
dbutils.widgets.text("pipeline_id", "", "6 Pipeline id (to refresh gold MV; optional)")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA  = dbutils.widgets.get("schema").strip()
FQ      = f"{CATALOG}.{SCHEMA}"
TOKEN   = dbutils.widgets.get("redaction_token")
DRY     = dbutils.widgets.get("dry_run") == "true"
PURGE   = dbutils.widgets.get("do_purge") == "true"
PIPELINE_ID = dbutils.widgets.get("pipeline_id").strip()

BASE_TABLES = ["silver_user", "bronze_user", "raw_user"]  # erase top-down; gold MV refreshed after
MV = "gold_user"
print("Schema:", FQ, "| dry_run:", DRY, "| purge:", PURGE)

## 1. Load the PENDING DSAR requests

Each row names one subject (email) and a request_type. We process them all in this
run — the multi-subject behavior you get in the base layer.

In [ ]:
from pyspark.sql import functions as F

def sqlstr(s):
    return "'" + str(s).replace("'", "''") + "'"

reqs = (spark.table(f"{FQ}.dsar_request")
        .where(F.col("status") == "PENDING")
        .select("request_id", "subject_email", "request_type")
        .collect())

if not reqs:
    print("No PENDING requests. Nothing to do.")
else:
    print(f"{len(reqs)} PENDING request(s):")
    for r in reqs:
        print(f"  {r['request_id']}  {r['subject_email']:<32} {r['request_type']}")

## 2. Resolve each subject email → user_id (from raw)

`raw_user` still holds cleartext email. We build, per request, the set of
`user_id`s to erase. (A subject with no match in raw was likely already erased.)

In [ ]:
plan = []   # list of dicts: {request_id, email, rtype, user_ids}
for r in reqs:
    email = r["subject_email"].strip().lower()
    ids = [x["user_id"] for x in
           spark.table(f"{FQ}.raw_user").where(F.lower("email") == email)
                .select("user_id").distinct().collect()]
    plan.append({"request_id": r["request_id"], "email": email,
                 "rtype": r["request_type"].strip().upper(), "user_ids": ids})
    print(f"  {r['request_id']}  {email:<32} {r['request_type']:<9} -> user_id(s): {ids or '(none)'}")

all_ids = sorted({u for p in plan for u in p["user_ids"]})
print("\nDistinct user_id(s) across all requests:", all_ids or "(none)")

## 3. Pre-count matches at every layer

So the erasure is auditable. Every table keys on `user_id`.

In [ ]:
if not all_ids:
    print("Nothing to erase.")
else:
    in_all = ", ".join(sqlstr(u) for u in all_ids)
    print("Rows matching any requested subject, per layer:")
    for t in BASE_TABLES + [MV]:
        try:
            n = spark.sql(f"SELECT count(*) c FROM {FQ}.{t} WHERE user_id IN ({in_all})").collect()[0]["c"]
            kind = "(MV, derived)" if t == MV else "(base table)"
            print(f"  {t:<14} {n}   {kind}")
        except Exception as e:
            print(f"  {t:<14} (missing) {str(e).splitlines()[0][:50]}")

## 4. Erase every subject from the base tables

Per request: **DELETE** removes the subject's rows; **OBFUSCATE** redacts PII cells
(only `raw_user` still has cleartext PII). Top-down silver → bronze → raw. Each
statement is a non-append commit the streams skip via `skipChangeCommits`. We erase
`raw_user` too, so a later full refresh can't resurrect the subject.

In [ ]:
def _mask_json(col):
    e = f"regexp_replace({col}, '(\"email\" *: *)\"[^\"]*\"', '$1\"{TOKEN}\"')"
    e = f"regexp_replace({e}, '(\"name\" *: *)\"[^\"]*\"', '$1\"{TOKEN}\"')"
    return e

modified = set()

def erase_stmt(table, user_ids, rtype):
    ul = ", ".join(sqlstr(u) for u in user_ids)
    pred = f"user_id IN ({ul})"
    if rtype == "DELETE":
        return f"DELETE FROM {FQ}.{table} WHERE {pred}"
    if table == "raw_user":   # OBFUSCATE: only raw still holds cleartext PII
        sets = [f"email = {sqlstr(TOKEN)}", f"full_name = {sqlstr(TOKEN)}",
                f"profile_json = {_mask_json('profile_json')}"]
        return f"UPDATE {FQ}.{table} SET {', '.join(sets)} WHERE {pred}"
    return None   # downstream already masked in OBFUSCATE mode

for p in plan:
    if not p["user_ids"]:
        print(f"[skip] {p['request_id']} {p['email']}: no rows (already erased?)")
        continue
    print(f"\n=== {p['request_id']}  {p['email']}  ({p['rtype']}) ===")
    for t in BASE_TABLES:
        stmt = erase_stmt(t, p["user_ids"], p["rtype"])
        if stmt is None:
            print(f"  [skip] {t}: already masked downstream (OBFUSCATE)")
            continue
        print(f"  [{'DRY' if DRY else 'RUN'}] {stmt}")
        if not DRY:
            spark.sql(stmt); modified.add(t)

print("\nBase tables modified:", sorted(modified) or "(none / dry run)")

## 5. Physical purge (CCPA "no trace")

Compact + VACUUM the base tables we modified. Serverless-safe technique (same as
base `03_physical_purge`): set table property
`delta.deletedFileRetentionDuration='interval 0 hours'` then a plain `VACUUM`
(`RETAIN 0 HOURS` trips the safety check on serverless).

In [ ]:
def purge(table):
    fqt = f"{FQ}.{table}"
    spark.sql(f"ALTER TABLE {fqt} SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = 'interval 0 hours')")
    try:
        spark.sql(f"REORG TABLE {fqt} APPLY (PURGE)")
    except Exception as e:
        print(f"  REORG skipped on {table}:", str(e).splitlines()[0][:70])
    spark.sql(f"VACUUM {fqt}")
    print(f"  purged {table}")

if DRY:
    print("Dry run — skipping purge.")
elif not PURGE:
    print("do_purge=false — skipping VACUUM.")
elif not modified:
    print("Nothing modified — skipping purge.")
else:
    for t in sorted(modified):
        try: purge(t)
        except Exception as e: print(f"  purge failed on {t}:", str(e).splitlines()[0][:70])

## 6. Refresh the gold materialized view

`gold_user` is a view derived from silver — we can't DELETE from it, and its stored
result still holds erased subjects until recomputed. Trigger a pipeline update so
it re-derives from the now-erased silver.

Set the `pipeline_id` widget to auto-refresh here; otherwise refresh from the
pipeline UI (Start), or `databricks pipelines start-update <id> --full-refresh`.

In [ ]:
if DRY:
    print("Dry run — gold MV not refreshed.")
elif not PIPELINE_ID:
    print("No pipeline_id set. Refresh gold_user by running the pipeline:")
    print("  UI: open the pipeline -> Start   (or Full refresh)")
    print("  CLI: databricks pipelines start-update <id> --full-refresh")
    print("Until refreshed, gold_user still shows erased subjects' aggregate rows.")
else:
    from databricks.sdk import WorkspaceClient
    w = WorkspaceClient()
    upd = w.pipelines.start_update(pipeline_id=PIPELINE_ID, full_refresh_selection=[MV])
    print(f"Triggered refresh of {MV}: {upd}. Wait for completion, then re-run section 7.")

## 7. Validate — no trace, and mark requests COMPLETE

Base-table counts must be **0** for DELETEd subjects; `gold_user` becomes 0 after
the MV refresh (section 6). We assert no cleartext email survives, then flip each
processed request to COMPLETE.

In [ ]:
if DRY:
    print("Dry run — no validation / status change. Re-run with dry_run=false.")
elif not reqs:
    print("Nothing was erased.")
else:
    if all_ids:
        in_all = ", ".join(sqlstr(u) for u in all_ids)
        print("Post-erasure counts (any requested subject):")
        for t in BASE_TABLES + [MV]:
            n = spark.sql(f"SELECT count(*) c FROM {FQ}.{t} WHERE user_id IN ({in_all})").collect()[0]["c"]
            note = "  <- refresh gold MV (section 6) to zero this" if (t == MV and n) else ""
            print(f"  {t:<14} {n}{note}")

    # --- DELETE subjects: no cleartext email + gone from every base table ---
    for p in plan:
        if p["rtype"] != "DELETE" or not p["user_ids"]:
            continue
        left = spark.sql(f"SELECT count(*) c FROM {FQ}.raw_user WHERE lower(email) = {sqlstr(p['email'])}").collect()[0]["c"]
        assert left == 0, f"cleartext {p['email']} survived in raw_user!"
        ul = ", ".join(sqlstr(u) for u in p["user_ids"])
        for t in BASE_TABLES:
            n = spark.sql(f"SELECT count(*) c FROM {FQ}.{t} WHERE user_id IN ({ul})").collect()[0]["c"]
            assert n == 0, f"{p['request_id']}: {t} still has {n} rows!"

    # --- OBFUSCATE subjects: rows kept, but PII redacted in raw_user (no cleartext trace) ---
    for p in plan:
        if p["rtype"] != "OBFUSCATE" or not p["user_ids"]:
            continue
        ul = ", ".join(sqlstr(u) for u in p["user_ids"])
        # original cleartext email must be gone
        bad_email = spark.sql(
            f"SELECT count(*) c FROM {FQ}.raw_user WHERE lower(email) = {sqlstr(p['email'])}").collect()[0]["c"]
        assert bad_email == 0, f"{p['request_id']}: cleartext email {p['email']} survived OBFUSCATE in raw_user!"
        # every kept row must have PII cells + in-JSON PII redacted to the token
        unmasked = spark.sql(
            f"SELECT count(*) c FROM {FQ}.raw_user WHERE user_id IN ({ul}) AND "
            f"(email <> {sqlstr(TOKEN)} OR full_name <> {sqlstr(TOKEN)} "
            f"OR profile_json LIKE '%' || {sqlstr(p['email'])} || '%' "     # original email must be gone from the JSON blob
            f"OR profile_json NOT LIKE '%' || {sqlstr(TOKEN)} || '%')"      # token must be present (masking ran)
            ).collect()[0]["c"]
        assert unmasked == 0, f"{p['request_id']}: {unmasked} raw_user row(s) still hold un-redacted PII after OBFUSCATE!"

    # --- mark ALL processed requests COMPLETE (incl. no-match: already satisfied) ---
    done_ids = [p["request_id"] for p in plan]          # every PENDING request handled this run
    nomatch  = [p["request_id"] for p in plan if not p["user_ids"]]
    if done_ids:
        idlist = ", ".join(sqlstr(i) for i in done_ids)
        spark.sql(f"UPDATE {FQ}.dsar_request SET status='COMPLETE' WHERE request_id IN ({idlist})")
    print("\nPASS — DELETE subjects erased with no trace; OBFUSCATE subjects redacted in place.")
    print("Requests marked COMPLETE:", done_ids)
    if nomatch:
        print("  (of those, no rows to erase — already satisfied:", nomatch, ")")
    display(spark.table(f"{FQ}.dsar_request"))


## 8. Idempotency

Because erasure removes subjects from the **base tables** (raw/bronze/silver), the
pipeline is idempotent under any mix of incremental and full-refresh updates: a
full refresh re-reads `raw_user` (subject already gone), so erased subjects never
reappear, and repeated updates converge to the same state. Re-running this notebook
is a no-op once the queue is COMPLETE.

In [ ]:
if not DRY and all_ids:
    in_all = ", ".join(sqlstr(u) for u in all_ids)
    print("Idempotency check — subject rows per layer:")
    for t in BASE_TABLES + [MV]:
        n = spark.sql(f"SELECT count(*) c FROM {FQ}.{t} WHERE user_id IN ({in_all})").collect()[0]["c"]
        flag = "  <- run gold MV refresh (section 6)" if (t == MV and n) else ""
        print(f"  {t:<14} {n}{flag}")
else:
    print("Dry run — idempotency check skipped.")